# Обнаружение фишинговых сайтов по признакам URL

Командный проект курса «Основы ИИ в кибербезопасности».

**Цель:** сравнить базовую и основные модели, оценить качество, устойчивость и объяснимость и сделать вывод об применимости в реальном SOC.

**Ограничения:** используются только открытые или синтетические датасеты; код не делает сетевых запросов к URL.

## Реальный датасет и фактические результаты

Используется открытый PhiUSIIL (UCI, 235795 строк, лицензия CC BY 4.0). Разделение: 80/20, seed 42, стратификация.

На отложенной выборке Random Forest: accuracy 1.000000, precision 1.000000, recall 1.000000, F1 1.000000, матрица ошибок [[20189, 0], [0, 26970]].

Logistic Regression: accuracy 0.999873, precision 0.999778, recall 1.000000, F1 0.999889, матрица ошибок [[20183, 6], [0, 26970]].

MLP: accuracy 0.999936, precision 0.999889, recall 1.000000, F1 0.999944, матрица ошибок [[20186, 3], [0, 26970]].

Указанные значения сохранены в `reports/phiusiil_metrics.json`. Почти идеальные метрики требуют проверки утечки признаков и независимости источников данных.

In [ ]:
# Загрузка фактических результатов реального датасета
REAL_METRICS_PATH = "../reports/phiusiil_metrics.json"
if os.path.exists(REAL_METRICS_PATH):
    import json
    with open(REAL_METRICS_PATH, "r", encoding="utf-8") as f:
        real_metrics = json.load(f)
    display(pd.DataFrame(real_metrics["models"]).T)
    print("Размер датасета:", real_metrics["instances_after_deduplication"])
    print("Seed:", real_metrics["random_seed"])
else:
    print("Сначала запустите: .venv/bin/python src/evaluate_real_dataset.py")

In [ ]:
# Оставляем исходный DATA_PATH согласованным с реальным датасетом.
DATA_PATH = "../data/raw/PhiUSIIL_Phishing_URL_Dataset.csv"
TARGET_COLUMN = "label"

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc,
    precision_recall_curve, average_precision_score
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
print("Seed:", RANDOM_STATE)

## 1. Загрузка открытого датасета

Укажите в переменной `DATA_PATH` путь к CSV. Сохраните источник и лицензию датасета в отчёте.

Ожидаемый столбец метки: `label`, где допустимо использовать строковые значения (`legitimate`, `phishing`) или `0/1`. Перед запуском проверьте код ниже и переименуйте целевой столбец при необходимости.

In [ ]:
DATA_PATH = "../data/raw/phishing_urls.csv"
TARGET_COLUMN = "label"

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print("Размер датасета:", df.shape)
    display(df.head())
else:
    df = None
    print("Добавьте открытый или синтетический датасет в data/raw/phishing_urls.csv")

if df is not None:
    print("Столбцы:", df.columns.tolist())
    print("Пропуски:", df.isna().sum().loc[lambda s: s > 0].to_dict())

## 2. Предобработка и EDA

Выполняется удаление явных дубликатов и проверка распределения классов. Затем числовые признаки масштабируются, а категориальные передаются препроцессору. Примеры реальных URL не открываются.

In [ ]:
if df is not None:
    df = df.drop_duplicates()
    print("Распределение классов:")
    display(df[TARGET_COLUMN].value_counts(dropna=False))

    plt.figure(figsize=(6, 4))
    sns.countplot(data=df, x=TARGET_COLUMN)
    plt.title("Распределение классов")
    plt.tight_layout()
    plt.show()

    numeric_df = df.select_dtypes(include="number")
    if numeric_df.shape[1] >= 2:
        plt.figure(figsize=(8, 6))
        sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0)
        plt.title("Корреляция числовых признаков")
        plt.tight_layout()
        plt.show()

## 3. Baseline-модель

Для базовой линии используется Logistic Regression. Разделение производится один раз с фиксированным seed и stratify по целевому столбцу.

In [ ]:
results = {}
models = {}

if df is not None:
    X = df.drop(columns=[TARGET_COLUMN])
    y = df[TARGET_COLUMN]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )

    numeric_columns = X.select_dtypes(include="number").columns.tolist()
    categorical_columns = X.select_dtypes(exclude="number").columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_columns),
        ("categorical", "passthrough", categorical_columns),
    ])

    classifiers = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
        "MLP": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=RANDOM_STATE),
    }

    for name, classifier in classifiers.items():
        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", classifier),
        ])
        pipeline.fit(X_train, y_train)
        pred = pipeline.predict(X_test)

        if len(np.unique(y_test)) == 2:
            positive = pred == 1 if set(np.unique(y_test)).issubset({0, 1}) else pred == sorted(np.unique(y_test))[1]
            precision = precision_score(y_test, pred, pos_label=1 if set(np.unique(y_test)).issubset({0, 1}) else sorted(np.unique(y_test))[1], zero_division=0)
            recall = recall_score(y_test, pred, pos_label=1 if set(np.unique(y_test)).issubset({0, 1}) else sorted(np.unique(y_test))[1], zero_division=0)
            f1 = f1_score(y_test, pred, pos_label=1 if set(np.unique(y_test)).issubset({0, 1}) else sorted(np.unique(y_test))[1], zero_division=0)
        else:
            precision = precision_score(y_test, pred, average="weighted", zero_division=0)
            recall = recall_score(y_test, pred, average="weighted", zero_division=0)
            f1 = f1_score(y_test, pred, average="weighted", zero_division=0)

        results[name] = {
            "accuracy": accuracy_score(y_test, pred),
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }
        models[name] = pipeline

    display(pd.DataFrame(results).T)

## 4–5. Сравнение моделей и визуализация метрик

Оцениваются Logistic Regression как baseline, Random Forest и MLP. Метрики считаются только на отложенной выборке.

In [ ]:
if df is not None:
    fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 5))
    if len(models) == 1:
        axes = [axes]

    for ax, (name, pipeline) in zip(axes, models.items()):
        pred = pipeline.predict(X_test)
        cm = confusion_matrix(y_test, pred)
        ConfusionMatrixDisplay(cm).plot(ax=ax, cmap="Blues", colorbar=False)
        ax.set_title(name)

    plt.tight_layout()
    plt.show()

    metric_frame = pd.DataFrame(results).T
    metric_frame.plot(kind="bar", figsize=(10, 5))
    plt.ylim(0, 1)
    plt.title("Сравнение метрик на отложенной выборке")
    plt.ylabel("Значение")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## 6. Устойчивость и объяснимость

Adversarial-примеры создаются синтетически без обращения к реальным доменам. Для каждого варианта URL меняются длина, знаки или содержащиеся слова; проверяется изменение предсказания и вероятности модели.

In [ ]:
# Пример безопасной синтетической проверки признаков
synthetic_urls = pd.DataFrame([
    {"url": "https://safe-example.test/login", "variant": "base"},
    {"url": "https://safe-example.test/login-confirm-account", "variant": "added_suspicious_words"},
    {"url": "https://192.0.2.10/login?id=123", "variant": "synthetic_ip_host"},
])
display(synthetic_urls)

# Для реального датасета после обучения Random Forest можно построить permutation importance.
if df is not None:
    from sklearn.inspection import permutation_importance

    best_name = max(results, key=lambda name: results[name]["f1"])
    print("Модель с лучшим F1:", best_name)
    importance = permutation_importance(
        models[best_name], X_test, y_test,
        scoring="f1_weighted", n_repeats=10,
        random_state=RANDOM_STATE, n_jobs=-1
    )

    feature_names = X_test.columns
    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance_mean": importance.importances_mean,
        "importance_std": importance.importances_std,
    }).sort_values("importance_mean", ascending=False)

    display(importance_df.head(20))
    plt.figure(figsize=(9, 6))
    sns.barplot(data=importance_df.head(20), x="importance_mean", y="feature")
    plt.title(f"Permutation importance: {best_name}")
    plt.tight_layout()
    plt.show()

    # SHAP можно подключить для модели Random Forest после проверки совместимости версий:
    # import shap
    # explainer = shap.Explainer(models["Random Forest"].named_steps["classifier"])
    # ...

## 7. Интерпретация для SOC

Заполнить результаты на основании отложенной выборки:

- FP (легитимные URL ошибочно как фишинг) создают шум для аналитиков и блокировки легитимных сервисов;
- FN (фишинг пропущен) несут более высокий риск компрометации учетных данных;
- порог вероятности должен выбираться с учётом стоимости ошибок и политиками SOC;
- система должна работать как помощник приоритизации, а не как единственный источник блокировки;
- нужны мониторинг дрейфа, переобучение по временным периодам и ручная проверка сомнительных кейсов;
- модель не должна обращаться к подозрительным доменам для feature extraction.

## 8. Итоговый отчёт и вклад участников

Собрать графики в `reports/`, метрики в CSV/JSON и заполнить таблицу вклада каждого участника.

In [ ]:
team_contributions = pd.DataFrame([
    {"Участник": "Участник 1", "Роль": "Data engineer", "Вклад": "Датасет, очистка, признаки", "Доля, %": 25},
    {"Участник": "Участник 2", "Роль": "ML engineer", "Вклад": "Baseline и основные модели", "Доля, %": 25},
    {"Участник": "Участник 3", "Роль": "Evaluation & XAI analyst", "Вклад": "Метрики, матрица ошибок, устойчивость, SHAP", "Доля, %": 25},
    {"Участник": "Участник 4", "Роль": "Security analyst / team lead", "Вклад": "SOC-анализ, отчёт, презентация, сроки", "Доля, %": 25},
])
display(team_contributions)

if df is not None:
    pd.DataFrame(results).T.to_csv("../reports/model_metrics.csv")
    print("Метрики сохранены в reports/model_metrics.csv")